<a href="https://colab.research.google.com/github/dkngit55-glitch/DKNGIT5/blob/main/LHJ_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U google-genai
!pip install -q pandas openpyxl
!pip install -q Pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 17.3 MB/s eta 0:00:00


In [4]:
import imaplib
import smtplib
import email
import re
import io
import pandas as pd
from PIL import Image
from email.header import decode_header
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google import genai
from google.colab import userdata

# --- [STEP 1] 설정 및 초기화 ---
NAVER_ID = "dkngit5"
NAVER_PW = "HCSHSG79ZBY8"
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

def safe_decode(header_value):
    if not header_value: return ""
    decoded_list = decode_header(header_value)
    result = ""
    for content, charset in decoded_list:
        if isinstance(content, bytes):
            try:
                result += content.decode(charset if charset else "utf-8", errors="ignore")
            except:
                result += content.decode("cp949", errors="ignore")
        else:
            result += content
    return result

def send_reply(to_email, subject, body):
    """답장 발송 함수"""
    try:
        smtp = smtplib.SMTP_SSL("smtp.naver.com", 465)
        smtp.login(f"{NAVER_ID}@naver.com", NAVER_PW)
        msg = MIMEMultipart()
        msg["From"] = f"{NAVER_ID}@naver.com"
        msg["To"] = to_email
        msg["Subject"] = "Re: " + subject
        msg.attach(MIMEText(body, "plain", "utf-8"))
        smtp.sendmail(f"{NAVER_ID}@naver.com", to_email, msg.as_string())
        smtp.quit()
        return True
    except Exception as e:
        print(f"❌ 발송 실패 ({to_email}): {e}")
        return False

# --- [STEP 2] 통합 에이전트 실행 (N개 메일 처리) ---
def run_batch_email_agent():
    print(f"⚡ {MODEL_ID} 기반 '읽지 않은 메일' 일괄 처리 에이전트 가동...")

    try:
        mail = imaplib.IMAP4_SSL("imap.naver.com")
        mail.login(f"{NAVER_ID}@naver.com", NAVER_PW)
        mail.select("inbox")

        # [변경 포인트] "ALL" 대신 "UNSEEN"으로 검색
        status, messages = mail.search(None, "UNSEEN")
        unread_ids = messages[0].split()

        total_count = len(unread_ids)
        if total_count == 0:
            print("📭 읽지 않은 새로운 메일이 없습니다.")
            mail.logout()
            return

        print(f"📂 총 {total_count}개의 읽지 않은 메일을 발견했습니다. 처리를 시작합니다.")

        # 각 메일에 대해 루프 실행
        for i, email_id in enumerate(unread_ids):
            print(f"\n[{i+1}/{total_count}] 메일 분석 중...")

            # 메일 데이터 가져오기
            status, msg_data = mail.fetch(email_id, "(RFC822)")

            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject = safe_decode(msg.get("Subject"))
                    from_raw = msg.get("From")
                    email_match = re.search(r'[\w\.-]+@[\w\.-]+', from_raw)
                    sender_email = email_match.group() if email_match else from_raw

                    excel_df = None
                    image_pil = None
                    plain_text_body = ""

                    # 첨부파일 및 본문 파싱
                    if msg.is_multipart():
                        for part in msg.walk():
                            content_type = part.get_content_type()
                            filename = safe_decode(part.get_filename())
                            if content_type == "text/plain" and not filename:
                                charset = part.get_content_charset() or 'utf-8'
                                plain_text_body = part.get_payload(decode=True).decode(charset, errors="ignore")
                            elif filename and (filename.lower().endswith('.xlsx') or filename.lower().endswith('.xls')):
                                excel_df = pd.read_excel(io.BytesIO(part.get_payload(decode=True)))
                            elif filename and (filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg')):
                                image_pil = Image.open(io.BytesIO(part.get_payload(decode=True)))

                    # AI 답장 생성 로직
                    reply_draft = ""
                    def generate_ai_reply(prompt, contents=None):
                        res = client.models.generate_content(model=MODEL_ID, contents=[prompt] + (contents or []))
                        return res.text.strip()

                    if excel_df is not None:
                        prompt = f"엑셀 데이터를 분석해서 정중한 '해요체' 답장을 써줘.\n[제목]:{subject}\n[데이터]:\n{excel_df.head(10).to_string()}"
                        reply_draft = generate_ai_reply(prompt)
                    elif image_pil is not None:
                        prompt = f"이미지 내용을 OCR 분석해서 정중한 '해요체' 답장을 써줘.\n[제목]:{subject}"
                        reply_draft = generate_ai_reply(prompt, contents=[image_pil])
                    else:
                        prompt = f"다음 메일 내용을 요약하고 '해요체'로 정중한 답장을 써줘.\n[제목]:{subject}\n[본문]:{plain_text_body[:500]}"
                        reply_draft = generate_ai_reply(prompt)

                    # 답장 전송
                    if reply_draft:
                        print(f"📧 보낸이: {sender_email}\n🤖 답장 초안 요약: {reply_draft[:50]}...")
                        if send_reply(sender_email, subject, reply_draft):
                            print(f"✅ 성공적으로 답장을 보냈습니다.")

        mail.logout()
        print(f"\n✨ 모든 작업이 완료되었습니다. (총 {total_count}건)")

    except Exception as e:
        print(f"❌ 중대한 오류 발생: {e}")

# 실행
run_batch_email_agent()

⚡ gemini-2.5-flash 기반 '읽지 않은 메일' 일괄 처리 에이전트 가동...
📂 총 2개의 읽지 않은 메일을 발견했습니다. 처리를 시작합니다.

[1/2] 메일 분석 중...
📧 보낸이: lhj@dong-il.com
🤖 답장 초안 요약: 공지사항1 잘 확인했어요.

중동 전쟁으로 인한 민생 안정 대책의 일환으로 유류세 인하 폭...
✅ 성공적으로 답장을 보냈습니다.

[2/2] 메일 분석 중...
📧 보낸이: lhj@dong-il.com
🤖 답장 초안 요약: 공지사항2 내용을 잘 확인했습니다.

미국·이스라엘과 이란 간의 중동 전쟁이 격화되면서 해...
✅ 성공적으로 답장을 보냈습니다.

✨ 모든 작업이 완료되었습니다. (총 2건)
